# DataMate — Vast notebook: pilot có cổng chất lượng rồi chạy đến public JSONL

Đặt `ACTION="run_all"` rồi chọn **Run All**. Pipeline chạy baseline 32 câu dev và nhãn pilot 32 câu train. Chỉ khi utility baseline không âm **và** ít nhất 12/32 plan được xác minh, notebook mới chạy toàn train, QLoRA, đánh giá dev và tạo `result/<RUN_ID>/submission.jsonl`.

Nếu một trong hai ngưỡng không đạt, notebook dừng trước full labeling/training để tránh đốt GPU vào baseline yếu. Mỗi bản pipeline có RUN_ID mới và cache nhãn có version riêng để không tái dùng pilot lỗi cũ. Full labeling vẫn có thể mất nhiều giờ.

In [ ]:
# Cấu hình run — chỉ đổi ACTION/RUN_ID và các giới hạn cần thiết ở đây.
from pathlib import Path
import hashlib
import json
import os
import shutil
import subprocess
import sys
import time
import zipfile

PROJECT_DIR = Path.cwd()
if not (PROJECT_DIR / "src/datamate").is_dir():
    candidates = [PROJECT_DIR.parent, PROJECT_DIR / "datamate", PROJECT_DIR.parent / "datamate",
                  PROJECT_DIR.parent.parent / "datamate"]
    PROJECT_DIR = next((path for path in candidates if (path / "src/datamate").is_dir()), PROJECT_DIR)
assert (PROJECT_DIR / "src/datamate").is_dir(), f"Mở notebook từ thư mục datamate. CWD={Path.cwd()}"

ACTION = "run_all"  # run_all = pipeline đầy đủ trong một lần; các action riêng vẫn dùng được
RUN_ID = "datamate-qwen4b-runall-v4-20261107"  # resume phải dùng lại RUN_ID của run cũ
PLAN_LABEL_LIMIT = 0  # 0 = toàn bộ train; pilot: 128
RUN_ALL_AUTO_CONTINUE_AFTER_AUDIT = True  # run_all xuất audit rồi tiếp tục tự động
AUDIT_CONFIRMED = False  # chỉ đặt True cho ACTION="train" sau khi đã tự xem file audit
RUN_ALL_DEV_LIMIT = 32  # pilot baseline, tránh chạy hết 553 câu trước khi biết pipeline ổn
RUN_ALL_PLAN_PILOT = 32  # pilot nhãn train trước khi chạy toàn bộ
RUN_ALL_MIN_PILOT_VERIFIED = 12  # tối thiểu 37.5% pilot phải khớp đáp án train
RUN_ALL_MIN_BASELINE_UTILITY = 0.0  # không train tiếp nếu utility baseline dev âm
RUN_ALL_TRAIN_DEV_LIMIT = 64  # dev subset cố định cho early stopping và báo cáo nhanh
SMOKE_TRAIN_SAMPLES = 64
SMOKE_DEV_SAMPLES = 16

DATA_ROOT = PROJECT_DIR / "data/raw"
ARCHIVE_ROOT = PROJECT_DIR / "data/archives"
PROCESSED_DIR = PROJECT_DIR / "data/processed/sft"
CONFIG_PATH = PROJECT_DIR / "configs/baseline.json"
RUN_DIR = PROJECT_DIR / "outputs/runs" / RUN_ID
HF_HOME = PROJECT_DIR / "models/hf_cache"
RESULT_DIR = PROJECT_DIR / "result" / RUN_ID
for directory in (RUN_DIR, HF_HOME, RESULT_DIR):
    directory.mkdir(parents=True, exist_ok=True)
os.environ["HF_HOME"] = str(HF_HOME)
os.environ["HF_HUB_CACHE"] = str(HF_HOME / "hub")
os.environ["TOKENIZERS_PARALLELISM"] = "false"

CONFIG = json.loads(CONFIG_PATH.read_text(encoding="utf-8"))
GPU_ACTIONS = {"run_all", "baseline_evaluate", "build_plans", "smoke", "train", "resume", "evaluate", "public_predict"}
VALID_ACTIONS = GPU_ACTIONS | {"prepare", "review_plans", "augment", "backup"}
assert ACTION in VALID_ACTIONS, f"ACTION không hợp lệ: {ACTION}"
print({"project": str(PROJECT_DIR), "action": ACTION, "run_id": RUN_ID,
       "data_root": str(DATA_ROOT), "run_dir": str(RUN_DIR), "hf_cache": str(HF_HOME)})

## 1. Môi trường và GPU

Notebook kiểm tra CUDA trước khi tải model. Nếu chưa có PyTorch, ô preflight thử cài CUDA wheel cu128; nếu máy có PyTorch nhưng không thấy CUDA, nó dừng để tránh train nhầm trên CPU. Không đóng kernel trong lúc cài hoặc tải model.


In [ ]:
import importlib.metadata
import importlib.util

needs_model_packages = ACTION in GPU_ACTIONS or ACTION == "backup"
if importlib.util.find_spec("torch") is None and ACTION in GPU_ACTIONS:
    subprocess.run([sys.executable, "-m", "pip", "install", "torch",
                    "--index-url", "https://download.pytorch.org/whl/cu128"], check=True)
if importlib.util.find_spec("torch") is not None:
    import torch
else:
    torch = None
if needs_model_packages:
    subprocess.run([sys.executable, "-m", "pip", "install", "-r", str(PROJECT_DIR / "requirements-train.txt"),
                    "--disable-pip-version-check"], check=True)
else:
    subprocess.run([sys.executable, "-m", "pip", "install", "pandas==2.3.3", "openpyxl==3.1.5",
                    "--disable-pip-version-check"], check=True)
if ACTION in GPU_ACTIONS and (torch is None or not torch.cuda.is_available()):
    raise RuntimeError("Không thấy CUDA từ PyTorch. Cài CUDA wheel phù hợp theo https://pytorch.org/get-started/locally/, restart kernel rồi chạy lại.")
print("Python:", sys.version)
print("PyTorch:", torch.__version__ if torch else "not installed (prepare/augment only)",
      "CUDA:", torch.version.cuda if torch else "n/a")
print("GPU:", torch.cuda.get_device_name(0) if torch and torch.cuda.is_available() else "không yêu cầu cho ACTION này")
if torch and torch.cuda.is_available():
    print("BF16 supported:", torch.cuda.is_bf16_supported())
    subprocess.run(["nvidia-smi"], check=False)
subprocess.run(["df", "-h", str(PROJECT_DIR)], check=False)
if needs_model_packages:
    for package in ("transformers", "peft", "accelerate", "bitsandbytes", "pandas", "openpyxl"):
        print(package, importlib.metadata.version(package))

## 2. Kiểm tra / giải nén dữ liệu

Giữ nguyên hai archive gốc. Nếu raw split chưa đủ file bắt buộc, notebook giải nén an toàn vào `data/raw/`, không ghi đè file đã tồn tại.

In [ ]:
import shutil

def safe_extract(zip_path: Path, destination: Path) -> None:
    destination.mkdir(parents=True, exist_ok=True)
    root = destination.resolve()
    with zipfile.ZipFile(zip_path) as archive:
        for member in archive.infolist():
            target = (destination / member.filename).resolve()
            if target != root and root not in target.parents:
                raise ValueError(f"Archive path không an toàn: {member.filename}")
            if member.is_dir():
                target.mkdir(parents=True, exist_ok=True)
                continue
            target.parent.mkdir(parents=True, exist_ok=True)
            if target.exists():
                continue
            with archive.open(member) as source, target.open("wb") as output:
                shutil.copyfileobj(source, output)

required = {
    "train": ["questions.jsonl", "answers.jsonl", "rulebook.md", "calendar.csv"],
    "public_test": ["questions.jsonl", "calendar.csv"],
}
for split, files in required.items():
    split_dir = DATA_ROOT / split
    missing = [name for name in files if not (split_dir / name).is_file()]
    if missing:
        archive_path = ARCHIVE_ROOT / f"{split}.zip"
        if not archive_path.is_file():
            raise FileNotFoundError(f"Thiếu {split}/{missing}; upload {archive_path} rồi chạy lại cell này.")
        print(f"Extracting {archive_path.name} -> {DATA_ROOT}")
        safe_extract(archive_path, DATA_ROOT)
    still_missing = [name for name in files if not (split_dir / name).is_file()]
    if still_missing:
        raise FileNotFoundError(f"Archive không có các file cần thiết: {split}/{still_missing}")
print("Train questions:", sum(1 for _ in open(DATA_ROOT / "train/questions.jsonl", encoding="utf-8")))
print("Public questions:", sum(1 for _ in open(DATA_ROOT / "public_test/questions.jsonl", encoding="utf-8")))

## 3. Chuẩn bị manifest, business-group split và inventory

Với `run_all`, cell này bảo đảm split đã có. Toàn bộ bước chạy sẽ được điều phối ở cell cuối; các cell action riêng ở giữa được bỏ qua.

In [ ]:
sys.path.insert(0, str(PROJECT_DIR / "src"))
from datamate.prepare_data import prepare_data

split_files = [PROCESSED_DIR / "train_questions.jsonl", PROCESSED_DIR / "dev_questions.jsonl"]
if ACTION == "prepare" or not all(path.is_file() for path in split_files):
    PREPARE_REPORT = prepare_data(DATA_ROOT, PROCESSED_DIR, CONFIG["dev_business_ids"])
    print(json.dumps(PREPARE_REPORT, ensure_ascii=False, indent=2))
else:
    split_manifest = json.loads((PROCESSED_DIR / "split_manifest.json").read_text(encoding="utf-8"))
    print("Đã có split:", {key: split_manifest[key] for key in ("train_count", "dev_count", "dev_businesses")})

## 4. Sinh plan đã xác minh (train only)

Bước này có thể chạy lâu. Checkpoint tiến độ được ghi mỗi câu để có thể chạy tiếp. Model **không nhận gold answer trong prompt**; gold chỉ được dùng sau khi executor chạy để quyết định giữ/bỏ plan.

In [ ]:
if ACTION == "build_plans":
    from datamate.modeling import load_model, load_tokenizer, set_seed
    from datamate.plan_labels import build_verified_plans
    set_seed(CONFIG["seed"])
    tokenizer = load_tokenizer(CONFIG["model_name"], CONFIG["model_revision"])
    teacher = load_model(CONFIG["model_name"], CONFIG["model_revision"], quantized=True)
    PLAN_REPORT = build_verified_plans(
        DATA_ROOT, PROCESSED_DIR, teacher, tokenizer,
        CONFIG["model_name"], CONFIG["model_revision"],
        max_samples=PLAN_LABEL_LIMIT, seed=CONFIG["seed"],
        max_new_tokens=CONFIG["max_new_tokens"], max_context_tokens=CONFIG["max_context_tokens"],
    )
    print(json.dumps(PLAN_REPORT, ensure_ascii=False, indent=2))
    del teacher
    torch.cuda.empty_cache()
else:
    print("Bỏ qua — đặt ACTION='build_plans' để chạy bước này.")

## 5. Audit thủ công nhãn verified

Plan chạy khớp gold vẫn có thể đúng do trùng hợp hoặc thiếu một rule chưa được executor mô hình hóa. Hãy mở file audit, đọc ít nhất 100 dòng đại diện, ưu tiên kiểu thời gian, doanh thu/trả hàng, top-k, list/table và unanswerable. Chỉ đặt `AUDIT_CONFIRMED=True` sau bước xem này.

In [ ]:
if ACTION == "review_plans":
    import random
    from datamate.io import read_jsonl, write_jsonl
    verified = read_jsonl(PROCESSED_DIR / "train_verified_plans.jsonl")
    sample_count = min(100, len(verified))
    audit_rows = random.Random(CONFIG["seed"]).sample(verified, sample_count)
    audit_path = RUN_DIR / "verified_plan_audit.jsonl"
    write_jsonl(audit_path, [{**row, "human_review": "pending"} for row in audit_rows])
    print(f"Audit file: {audit_path} | rows={sample_count}")
    import pandas as pd
    display(pd.DataFrame([{"id": row["id"], "business_id": row["business_id"],
                           "question": row["question"], "plan": json.dumps(row["plan"], ensure_ascii=False),
                           "prediction": row["prediction"], "gold": row["gold"]}
                          for row in audit_rows[:10]]))
else:
    print("Bỏ qua — đặt ACTION='review_plans' để xuất 100 plan cần xem.")

## 6. Augmentation train-only

Chỉ thêm một số biến thể viết lại xác định, không đổi ngày/entity/ý nghĩa. Bản dev không augmentation.

In [ ]:
if ACTION == "augment":
    from datamate.augment import augment_verified_plans
    AUGMENT_REPORT = augment_verified_plans(
        PROCESSED_DIR / "train_verified_plans.jsonl",
        PROCESSED_DIR / "train_augmented_plans.jsonl",
    )
    print(json.dumps(AUGMENT_REPORT, ensure_ascii=False, indent=2))
else:
    print("Bỏ qua — đặt ACTION='augment' để chạy bước này.")

## 7. Smoke train / full train / resume

- Smoke: 64 verified examples, 1 epoch, batch=1/accumulation=1, dev tối đa 16 câu. Xác nhận loss hữu hạn, adapter lưu được, end-to-end callback chạy được.
- Full: tối đa 5 epoch, QLoRA 4-bit NF4 + BF16 nếu GPU hỗ trợ, gradient checkpointing, LoRA rank 16; early stop dựa trên utility dev, không dựa riêng vào loss.
- Resume: giữ nguyên `RUN_ID`; script tự chọn checkpoint gần nhất hoặc dùng `--resume-from-checkpoint`.

In [ ]:
if ACTION in {"smoke", "train", "resume"}:
    verified_path = PROCESSED_DIR / "train_verified_plans.jsonl"
    augmented_path = PROCESSED_DIR / "train_augmented_plans.jsonl"
    if not verified_path.is_file():
        raise FileNotFoundError("Chưa có plan verified. Chạy ACTION='build_plans' trước.")
    if augmented_path.is_file():
        from datamate.io import read_jsonl
        current_ids = {row["id"] for row in read_jsonl(verified_path)}
        augmented_rows = read_jsonl(augmented_path)
        augmented_parent_ids = {row["id"] for row in augmented_rows if "parent_id" not in row}
        if current_ids != augmented_parent_ids:
            raise ValueError("Augmentation không khớp bộ verified hiện tại; chạy ACTION='augment' lại.")
    train_path = augmented_path if augmented_path.is_file() else verified_path
    verified_count = sum(1 for _ in open(verified_path, encoding="utf-8"))
    if ACTION == "train" and not AUDIT_CONFIRMED:
        raise ValueError("Đọc verified_plan_audit.jsonl trước; sau khi xem ít nhất 100 dòng, đặt AUDIT_CONFIRMED=True.")
    if ACTION == "train" and verified_count < 128:
        raise ValueError(f"Chỉ có {verified_count} nhãn verified; xem plan_generation_report.json và sửa pipeline trước khi full train.")
    training_output = RUN_DIR / (f"smoke-{int(time.time())}" if ACTION == "smoke" else "qlora")
    command = [sys.executable, str(PROJECT_DIR / "training/train_lora.py"),
               "--config", str(CONFIG_PATH), "--data-root", str(DATA_ROOT),
               "--train-jsonl", str(train_path), "--dev-jsonl", str(PROCESSED_DIR / "dev_questions.jsonl"),
               "--output-dir", str(training_output)]
    if ACTION == "smoke":
        command += ["--max-train-samples", str(SMOKE_TRAIN_SAMPLES),
                    "--max-dev-samples", str(SMOKE_DEV_SAMPLES), "--epochs", "1",
                    "--micro-batch-size", "1", "--gradient-accumulation-steps", "1",
                    "--max-sequence-length", str(CONFIG["max_sequence_length"])]
    elif ACTION == "resume":
        command.append("--resume")
    env = os.environ.copy()
    env["PYTHONPATH"] = str(PROJECT_DIR / "src") + os.pathsep + env.get("PYTHONPATH", "")
    print("Command:", " ".join(command))
    subprocess.run(command, cwd=PROJECT_DIR, env=env, check=True)
else:
    print("Bỏ qua — chọn ACTION='smoke', 'train' hoặc 'resume'.")

## 8. Đánh giá dev hoặc tạo submission public

Dev report có utility sum/mean, đúng/sai/abstain/coverage. Public output theo từng dòng `id/type/answer`; lỗi schema/plan được trả abstain, không làm notebook dừng giữa chừng. Đối chiếu rule DataMate chính thức trước khi nộp.

In [ ]:
if ACTION in {"baseline_evaluate", "evaluate", "public_predict"}:
    adapter_path = RUN_DIR / "qlora/best-adapter" if ACTION != "baseline_evaluate" else None
    if ACTION != "baseline_evaluate" and not adapter_path.is_dir():
        raise FileNotFoundError(f"Không tìm thấy best adapter: {adapter_path}")
    env = os.environ.copy()
    env["PYTHONPATH"] = str(PROJECT_DIR / "src") + os.pathsep + env.get("PYTHONPATH", "")
    if ACTION in {"baseline_evaluate", "evaluate"}:
        eval_dir = RUN_DIR / ("dev_prompt_only" if ACTION == "baseline_evaluate" else "dev_final")
        command = [sys.executable, "-m", "datamate.cli", "evaluate", "--data-root", str(DATA_ROOT),
                   "--processed-dir", str(PROCESSED_DIR), "--config", str(CONFIG_PATH),
                   "--output-dir", str(eval_dir)]
        if adapter_path is not None:
            command += ["--adapter", str(adapter_path)]
    else:
        submission_path = RESULT_DIR / "submission.jsonl"
        command = [sys.executable, "-m", "datamate.cli", "public-predict", "--data-root", str(DATA_ROOT),
                   "--config", str(CONFIG_PATH), "--output", str(submission_path), "--adapter", str(adapter_path)]
    env["PYTHONPATH"] = str(PROJECT_DIR / "src") + os.pathsep + env.get("PYTHONPATH", "")
    subprocess.run(command, cwd=PROJECT_DIR, env=env, check=True)
else:
    print("Bỏ qua — chọn ACTION='baseline_evaluate', 'evaluate' hoặc 'public_predict'.")

## 9. Backup gói offline

Gói này gồm source DataMate, cấu hình, best adapter và base model đã cache để tránh phụ thuộc mạng lúc inference. Nó **không** chứa bộ train/public/private. Kích thước có thể vài GB; kiểm tra dung lượng còn trống trước khi chạy.

In [ ]:
if ACTION == "backup":
    from huggingface_hub import snapshot_download
    adapter_path = RUN_DIR / "qlora/best-adapter"
    if not adapter_path.is_dir():
        raise FileNotFoundError(f"Không tìm thấy adapter: {adapter_path}")
    base_snapshot = Path(snapshot_download(
        repo_id=CONFIG["model_name"], revision=CONFIG["model_revision"],
        cache_dir=str(HF_HOME / "hub"), local_files_only=True,
    ))
    handoff_dir = PROJECT_DIR / "outputs/handoff" / RUN_ID
    handoff_dir.mkdir(parents=True, exist_ok=True)
    archive_path = handoff_dir / "datamate_offline_bundle.tar"
    import tarfile
    with tarfile.open(archive_path, "w") as archive:
        archive.dereference = True
        archive.add(base_snapshot, arcname="base_model")
        archive.add(adapter_path, arcname="adapter")
        archive.add(PROJECT_DIR / "src/datamate", arcname="src/datamate")
        archive.add(PROJECT_DIR / "training/train_lora.py", arcname="training/train_lora.py")
        archive.add(PROJECT_DIR / "scripts/run_inference.py", arcname="scripts/run_inference.py")
        archive.add(CONFIG_PATH, arcname="configs/baseline.json")
        archive.add(PROJECT_DIR / "requirements-inference.txt", arcname="requirements-inference.txt")
    digest = hashlib.sha256()
    with archive_path.open("rb") as stream:
        for block in iter(lambda: stream.read(1024 * 1024), b""):
            digest.update(block)
    digest = digest.hexdigest()
    manifest = {"run_id": RUN_ID, "model": CONFIG["model_name"], "revision": CONFIG["model_revision"],
                "adapter": str(adapter_path), "base_snapshot": str(base_snapshot),
                "bundle": str(archive_path), "bundle_sha256": digest}
    (handoff_dir / "manifest.json").write_text(json.dumps(manifest, ensure_ascii=False, indent=2) + "\n", encoding="utf-8")
    print(json.dumps(manifest, ensure_ascii=False, indent=2))
else:
    print("Bỏ qua — chỉ chạy sau khi đã chọn best adapter và ACTION='backup'.")

## 10. Chạy pipeline có cổng pilot

Chạy baseline 32 mẫu và plan pilot 32 mẫu trước. Chỉ chạy toàn bộ train khi utility baseline không âm và ít nhất 12 plan được xác minh; nếu không, notebook dừng trước full labeling/training. Early stopping dùng 64 câu dev cố định.


In [ ]:
if ACTION == "run_all":
    import random
    from datamate.io import read_jsonl, write_jsonl

    env = os.environ.copy()
    env["PYTHONPATH"] = str(PROJECT_DIR / "src") + os.pathsep + env.get("PYTHONPATH", "")
    cli = [sys.executable, "-m", "datamate.cli"]

    def run_stage(label, command):
        print(f"\n===== {label} =====", flush=True)
        subprocess.run(command, cwd=PROJECT_DIR, env=env, check=True)

    run_stage("Prepare / split / inventory", cli + [
        "prepare", "--data-root", str(DATA_ROOT), "--processed-dir", str(PROCESSED_DIR),
        "--config", str(CONFIG_PATH),
    ])

    baseline_dir = RUN_DIR / "dev_prompt_pilot"
    baseline_metrics = baseline_dir / "metrics.json"
    if not baseline_metrics.is_file():
        run_stage("Prompt-only baseline on 32 grouped-dev examples", cli + [
            "evaluate", "--data-root", str(DATA_ROOT), "--processed-dir", str(PROCESSED_DIR),
            "--config", str(CONFIG_PATH), "--output-dir", str(baseline_dir),
            "--limit", str(RUN_ALL_DEV_LIMIT),
        ])
    else:
        print("Prompt-only pilot already exists; reusing it.")

    baseline_report = json.loads(baseline_metrics.read_text(encoding="utf-8"))
    baseline_utility = float(baseline_report.get("utility_mean", float("-inf")))
    print("Baseline utility:", baseline_utility, flush=True)
    if baseline_utility < RUN_ALL_MIN_BASELINE_UTILITY:
        raise RuntimeError(
            f"Baseline gate failed: utility_mean={baseline_utility:.4f}; "
            f"need >= {RUN_ALL_MIN_BASELINE_UTILITY:.4f}. No plan generation, full labeling, or training was started."
        )

    pilot_report_path = PROCESSED_DIR / "plan_generation_report.json"
    pilot_verified_path = PROCESSED_DIR / "train_verified_plans.jsonl"
    pilot_complete = False
    if pilot_report_path.is_file() and pilot_verified_path.is_file():
        previous = json.loads(pilot_report_path.read_text(encoding="utf-8"))
        pilot_complete = (previous.get("pipeline_version") == 2 and
                          previous.get("selected") == RUN_ALL_PLAN_PILOT and
                          previous.get("processed") == RUN_ALL_PLAN_PILOT)
    if not pilot_complete:
        run_stage("Build 32 train-only pilot plans", cli + [
            "build-plans", "--data-root", str(DATA_ROOT), "--processed-dir", str(PROCESSED_DIR),
            "--config", str(CONFIG_PATH), "--limit", str(RUN_ALL_PLAN_PILOT),
        ])
    pilot_report = json.loads(pilot_report_path.read_text(encoding="utf-8"))
    pilot_verified_count = len(read_jsonl(pilot_verified_path))
    print("Pilot report:", json.dumps(pilot_report, ensure_ascii=False), flush=True)
    if pilot_verified_count < RUN_ALL_MIN_PILOT_VERIFIED:
        raise RuntimeError(
            f"Pilot gate failed: {pilot_verified_count}/{RUN_ALL_PLAN_PILOT} verified; "
            f"need at least {RUN_ALL_MIN_PILOT_VERIFIED}. No full labeling or training was started."
        )

    expected_train_count = len(read_jsonl(PROCESSED_DIR / "train_questions.jsonl"))
    full_selection = min(PLAN_LABEL_LIMIT or expected_train_count, expected_train_count)
    full_report_path = PROCESSED_DIR / "plan_generation_report.json"
    full_complete = False
    if full_report_path.is_file() and pilot_report.get("selected") != full_selection:
        previous_full = json.loads(full_report_path.read_text(encoding="utf-8"))
        full_complete = (previous_full.get("pipeline_version") == 2 and
                         previous_full.get("selected") == full_selection and
                         previous_full.get("processed") == full_selection)
    if not full_complete:
        run_stage("Build all train-only verified plans", cli + [
            "build-plans", "--data-root", str(DATA_ROOT), "--processed-dir", str(PROCESSED_DIR),
            "--config", str(CONFIG_PATH), "--limit", str(PLAN_LABEL_LIMIT),
        ])

    plan_report = json.loads(full_report_path.read_text(encoding="utf-8"))
    verified_path = PROCESSED_DIR / "train_verified_plans.jsonl"
    verified = read_jsonl(verified_path)
    verified_count = len(verified)
    print("Full plan report:", json.dumps(plan_report, ensure_ascii=False), flush=True)
    if verified_count < 128:
        raise RuntimeError(f"Chỉ có {verified_count} verified plans; cần ít nhất 128 để train. Xem {full_report_path}.")

    audit_path = RUN_DIR / "verified_plan_audit.jsonl"
    audit_rows = random.Random(CONFIG["seed"]).sample(verified, min(100, verified_count))
    write_jsonl(audit_path, [{**row, "human_review": "pending"} for row in audit_rows])
    print(f"Saved {len(audit_rows)} audit examples to {audit_path}.", flush=True)
    if not RUN_ALL_AUTO_CONTINUE_AFTER_AUDIT:
        raise RuntimeError("Audit review gate is enabled. Review the file above before training.")
    print("Continuing without manual audit; review the audit file after this run.", flush=True)

    run_stage("Train-only augmentation", cli + [
        "augment", "--data-root", str(DATA_ROOT), "--processed-dir", str(PROCESSED_DIR),
        "--config", str(CONFIG_PATH),
    ])
    train_path = PROCESSED_DIR / "train_augmented_plans.jsonl"
    full_output = RUN_DIR / "qlora"
    training_report_path = full_output / "training_report.json"

    if not training_report_path.is_file():
        smoke_output = RUN_DIR / f"run_all_smoke-{int(time.time())}"
        smoke_command = [sys.executable, str(PROJECT_DIR / "training/train_lora.py"),
                         "--config", str(CONFIG_PATH), "--data-root", str(DATA_ROOT),
                         "--train-jsonl", str(train_path), "--dev-jsonl", str(PROCESSED_DIR / "dev_questions.jsonl"),
                         "--output-dir", str(smoke_output), "--max-train-samples", str(SMOKE_TRAIN_SAMPLES),
                         "--max-dev-samples", str(SMOKE_DEV_SAMPLES), "--epochs", "1",
                         "--micro-batch-size", "1", "--gradient-accumulation-steps", "1",
                         "--max-sequence-length", str(CONFIG["max_sequence_length"])]
        run_stage("QLoRA smoke run", smoke_command)

        train_command = [sys.executable, str(PROJECT_DIR / "training/train_lora.py"),
                         "--config", str(CONFIG_PATH), "--data-root", str(DATA_ROOT),
                         "--train-jsonl", str(train_path), "--dev-jsonl", str(PROCESSED_DIR / "dev_questions.jsonl"),
                         "--output-dir", str(full_output), "--max-dev-samples", str(RUN_ALL_TRAIN_DEV_LIMIT)]
        if (full_output / "data_manifest.json").is_file():
            train_command.append("--resume")
        run_stage("Full QLoRA with utility early stopping on 64 dev examples", train_command)
    else:
        print("Full training report exists; reusing its best adapter.")

    adapter_path = full_output / "best-adapter"
    if not adapter_path.is_dir():
        raise FileNotFoundError(f"Training did not produce best adapter: {adapter_path}")
    run_stage("Final grouped-dev evaluation on 64 examples", cli + [
        "evaluate", "--data-root", str(DATA_ROOT), "--processed-dir", str(PROCESSED_DIR),
        "--config", str(CONFIG_PATH), "--output-dir", str(RUN_DIR / "dev_final"),
        "--adapter", str(adapter_path), "--limit", str(RUN_ALL_TRAIN_DEV_LIMIT),
    ])
    submission_path = RESULT_DIR / "submission.jsonl"
    run_stage("Public-test prediction", cli + [
        "public-predict", "--data-root", str(DATA_ROOT), "--config", str(CONFIG_PATH),
        "--output", str(submission_path), "--adapter", str(adapter_path),
    ])
    print(f"\nRUN COMPLETE\nSubmission: {submission_path}\nAudit sample: {audit_path}", flush=True)
else:
    print("Bỏ qua — đặt ACTION='run_all' để chạy toàn bộ pipeline.")
